# 01 — Data Preparation
**Ubon Ratchathani Constituency 10** | Data Science Project 2025/2

This notebook:
1. Load + parse `data.json` and mapping files
2. Flatten into tidy DataFrames (constituency & party_list)
3. Tag metadata: amphoe, tambon, station_type, form_type
4. Validate & QA
5. Export to `data-processed/` for downstream notebooks

## 0 · Imports & Config

In [18]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

RAW = Path('../data-final')
OUT = Path('../data-processed/1')
OUT.mkdir(parents=True, exist_ok=True)

print('Paths OK:', RAW, OUT)

Paths OK: ../data-final ../data-processed/1


## 1 · Load Raw Files

In [19]:
# Main election data
with open(RAW / 'data.json', encoding='utf-8') as f:
    raw = json.load(f)

# Candidate mapping
with open(RAW / 'candidate_mapping_color.json', encoding='utf-8') as f:
    cand_map_raw = json.load(f)

# Party list mapping
with open(RAW / 'party_list_mapping.json', encoding='utf-8') as f:
    party_map_raw = json.load(f)

print(f'Loaded {len(raw):,} station records')
print(f'Candidate map: {len(cand_map_raw)} candidates')
print(f'Party map: {len(party_map_raw)} parties')

Loaded 313 station records
Candidate map: 6 candidates
Party map: 57 parties


## 2 · Build Lookup Dicts

In [20]:
# number to candidate info (constituency)
cand_map = {
    c['number']: {
        'candidate_name': c['candidate_name'],
        'party_name': c['party_name'],
        'color_hex': c['color_hex'],
    }
    for c in cand_map_raw
}

# number to party name (party list)
party_map = {p['number']: p['party_name'] for p in party_map_raw}

print('cand_map sample:', {k: v for k, v in list(cand_map.items())[:2]})
print('party_map sample:', {k: v for k, v in list(party_map.items())[:3]})

cand_map sample: {1: {'candidate_name': 'นายเกษม ฉิมพลี', 'party_name': 'ภูมิใจไทย', 'color_hex': '#212b6b'}, 2: {'candidate_name': 'นายสมศักดิ์ บุญประชม', 'party_name': 'เพื่อไทรวมพลัง', 'color_hex': '#ff66cc'}}
party_map sample: {1: 'ไทยทรัพย์ทวี', 2: 'เพื่อชาติไทย', 3: 'ใหม่'}


## 3 · Parse Station Metadata

Path structures found in the data:
- **depth 3** → advance out-of-district: `election_data / ล่วงหน้า... / ส.ส.5-17 (บช) ชุดที่ N`
- **depth 4** → election day (normal): `election_data / amphoe / tambon / หน่วยที่ N`
- **depth 5** → election day (form split): `election_data / amphoe / tambon / หน่วยที่ N / form_type`

In [21]:
def parse_station_meta(station_name: str) -> dict:
    """
    Parse station_name path → structured metadata dict
    Returns: amphoe, tambon, station_id, station_type, form_type
    """
    parts = station_name.split('\\')
    depth = len(parts)

    if depth == 3:
        raw_form = parts[2]  
        form_base = raw_form.split(' ชุดที่')[0].strip()
        is_party_list = '(บช)' in form_base or 'บช' in form_base
        return {
            'amphoe': 'ล่วงหน้านอกเขต',
            'tambon': 'ล่วงหน้านอกเขต',
            'station_id': raw_form,
            'station_type': 'advance_out',   # 5/17
            'form_type': 'party_list' if is_party_list else 'constituency',
        }

    elif depth == 4:
        return {
            'amphoe': parts[1],
            'tambon': parts[2],
            'station_id': parts[3],
            'station_type': 'election_day',  # 5/18
            'form_type': 'combined',
        }

    elif depth == 5:
        # some station split its form
        form_raw = parts[4]
        is_party_list = 'บช' in form_raw
        return {
            'amphoe': parts[1],
            'tambon': parts[2],
            'station_id': parts[3],
            'station_type': 'election_day',
            'form_type': 'party_list' if is_party_list else 'constituency',
        }

    else:
        return {
            'amphoe': 'UNKNOWN',
            'tambon': 'UNKNOWN',
            'station_id': station_name,
            'station_type': 'UNKNOWN',
            'form_type': 'UNKNOWN',
        }

# Quick test
test_cases = [
    raw[0]['station_name'],   # depth 3
    raw[20]['station_name'],  # depth 4
]
for tc in test_cases:
    print(tc)
    print(' →', parse_station_meta(tc))
    print()

election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราชอาณาจักร\ส.ส.5-17 (บช) ชุดที่ 1
 → {'amphoe': 'ล่วงหน้านอกเขต', 'tambon': 'ล่วงหน้านอกเขต', 'station_id': 'ส.ส.5-17 (บช) ชุดที่ 1', 'station_type': 'advance_out', 'form_type': 'party_list'}

election_data\อำเภอทุ่งศรีอุดม\ตำบลกุดเรือ\หน่วยที่ 9
 → {'amphoe': 'อำเภอทุ่งศรีอุดม', 'tambon': 'ตำบลกุดเรือ', 'station_id': 'หน่วยที่ 9', 'station_type': 'election_day', 'form_type': 'combined'}



## 4 · Flatten → Tidy DataFrames

Two output tables:
- `df_const` — constituency scores, one row per (station × candidate)
- `df_party` — party-list scores, one row per (station × party)

In [22]:
const_rows = []
party_rows = []

for record in raw:
    meta = parse_station_meta(record['station_name'])
    base = {
        'station_name': record['station_name'],
        'amphoe':       meta['amphoe'],
        'tambon':       meta['tambon'],
        'station_id':   meta['station_id'],
        'station_type': meta['station_type'],
    }

    # ── Constituency rows ──
    for r in record['constituency_results']:
        info = cand_map.get(r['number'], {})
        const_rows.append({
            **base,
            'cand_number':    r['number'],
            'candidate_name': info.get('candidate_name', f"เบอร์{r['number']}"),
            'party_name':     info.get('party_name',     'Unknown'),
            'color_hex':      info.get('color_hex',      '#cccccc'),
            'score':          r['score'],
        })

    # ── Party list rows ──
    for r in record['party_list_results']:
        party_rows.append({
            **base,
            'party_number': r['number'],
            'party_name':   party_map.get(r['number'], f"เบอร์{r['number']}"),
            'score':        r['score'],
        })

df_const = pd.DataFrame(const_rows)
df_party = pd.DataFrame(party_rows)

print(f'df_const shape: {df_const.shape}')
print(f'df_party shape: {df_party.shape}')
df_const.head(3)

df_const shape: (1812, 10)
df_party shape: (17043, 8)


,station_name,amphoe,tambon,station_id,station_type,cand_number,candidate_name,party_name,color_hex,score
0,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 ชุดที่ 9,advance_out,1,นายเกษม ฉิมพลี,ภูมิใจไทย,#212b6b,36
1,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 ชุดที่ 9,advance_out,2,นายสมศักดิ์ บุญประชม,เพื่อไทรวมพลัง,#ff66cc,257
2,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 ชุดที่ 9,advance_out,3,นายวัชรพล เชื้อคง,เพื่อไทย,#e0002b,4


In [23]:
df_party.head(3)

,station_name,amphoe,tambon,station_id,station_type,party_number,party_name,score
0,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 (บช) ชุดที่ 1,advance_out,1,ไทยทรัพย์ทวี,1
1,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 (บช) ชุดที่ 1,advance_out,2,เพื่อชาติไทย,12
2,election_data\ล่วงหน้านอกเขตเลือกตั้งและนอกราช...,ล่วงหน้านอกเขต,ล่วงหน้านอกเขต,ส.ส.5-17 (บช) ชุดที่ 1,advance_out,3,ใหม่,3


## 5 · Validate & QA

In [24]:
print('=' * 50)
print('QA: CONSTITUENCY')
print('=' * 50)

# 5.1 Total scores per candidate
total_const = (
    df_const.groupby(['cand_number', 'candidate_name', 'party_name'])['score']
    .sum()
    .reset_index()
    .sort_values('score', ascending=False)
)
print('\nคะแนนรวม ส.ส. เขต:')
print(total_const.to_string(index=False))

# 5.2 Check for missing scores
null_const = df_const['score'].isna().sum()
print(f'\nNull scores: {null_const}')

# 5.3 Stations with unexpected candidate count
cand_per_station = df_const.groupby('station_name')['cand_number'].nunique()
bad_stations = cand_per_station[cand_per_station != 6]
print(f'Stations NOT having 6 candidates: {len(bad_stations)}')
if len(bad_stations) > 0:
    print(bad_stations)

QA: CONSTITUENCY

คะแนนรวม ส.ส. เขต:
 cand_number          candidate_name     party_name  score
           2    นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง  60543
           3       นายวัชรพล เชื้อคง       เพื่อไทย   5647
           5    นายสิโรตม์ แสนทวีสุข        ประชาชน   4849
           1          นายเกษม ฉิมพลี      ภูมิใจไทย   3111
           4 นางสาวมิลิณ กำเนิดสิงห์   พลังประชารัฐ    887
           6            นายยิ่ง ภูผา   ประชาธิปัตย์    255

Null scores: 0
Stations NOT having 6 candidates: 0


In [25]:
print('=' * 50)
print('QA: PARTY LIST')
print('=' * 50)

# Top 10 parties by total score
total_party = (
    df_party.groupby(['party_number', 'party_name'])['score']
    .sum()
    .reset_index()
    .sort_values('score', ascending=False)
)
print('\nTop 10 พรรค (บัญชีรายชื่อ):')
print(total_party.head(10).to_string(index=False))

# Check null
null_party = df_party['score'].isna().sum()
print(f'\nNull scores: {null_party}')

# Stations with unexpected party count
party_per_station = df_party.groupby('station_name')['party_number'].nunique()
bad_p = party_per_station[party_per_station != 57]
print(f'Stations NOT having 57 parties: {len(bad_p)}')
if len(bad_p) > 0:
    print(bad_p.head())

QA: PARTY LIST

Top 10 พรรค (บัญชีรายชื่อ):
 party_number     party_name  score
           21      ไทรวมพลัง  39400
           46        ประชาชน   7588
            9       เพื่อไทย   7180
           57 พลังไทยรักชาติ   3418
           22      ก้าวอิสระ   2545
            2   เพื่อชาติไทย   2389
           56 เพื่อบ้านเมือง   2103
           20        ฟิวชั่น   1876
           48    ไทยสร้างไทย   1601
           37      ภูมิใจไทย   1557

Null scores: 0
Stations NOT having 57 parties: 0


In [26]:
print('=' * 50)
print('QA: STATION COVERAGE')
print('=' * 50)

# Station count per amphoe
station_counts = (
    df_const[['amphoe', 'tambon', 'station_name']]
    .drop_duplicates()
    .groupby('amphoe')
    .agg(tambon_count=('tambon', 'nunique'), station_count=('station_name', 'nunique'))
    .reset_index()
)
print('\nจำนวน station รายอำเภอ (constituency):')
print(station_counts.to_string(index=False))
print(f'\nTotal unique stations: {df_const["station_name"].nunique()}')

QA: STATION COVERAGE

จำนวน station รายอำเภอ (constituency):
          amphoe  tambon_count  station_count
  ล่วงหน้านอกเขต             1             24
อำเภอทุ่งศรีอุดม             5             54
    อำเภอน้ำขุ่น             4             47
     อำเภอน้ำยืน             7            104
      อำเภอสำโรง             6             73

Total unique stations: 302


## 6 · Create Aggregated Tables

สร้าง summary tables ที่ notebook ถัดๆ ไปจะใช้บ่อย

In [27]:
# Constituency scores by amphoe
const_by_amphoe = (
    df_const
    .groupby(['amphoe', 'cand_number', 'candidate_name', 'party_name', 'color_hex'])['score']
    .sum()
    .reset_index()
)

# Constituency scores by tambon
const_by_tambon = (
    df_const
    .groupby(['amphoe', 'tambon', 'cand_number', 'candidate_name', 'party_name', 'color_hex'])['score']
    .sum()
    .reset_index()
)

# Party list by amphoe (top 10)
top10_party_nums = total_party.head(10)['party_number'].tolist()
party_by_amphoe = (
    df_party[df_party['party_number'].isin(top10_party_nums)]
    .groupby(['amphoe', 'party_number', 'party_name'])['score']
    .sum()
    .reset_index()
)

# Total votes per station (for turnout later)
votes_per_station = (
    df_const
    .groupby(['amphoe', 'tambon', 'station_name', 'station_type'])['score']
    .sum()
    .reset_index()
    .rename(columns={'score': 'total_votes'})
)

print('const_by_amphoe:', const_by_amphoe.shape)
print('const_by_tambon:', const_by_tambon.shape)
print('party_by_amphoe:', party_by_amphoe.shape)
print('votes_per_station:', votes_per_station.shape)

const_by_amphoe: (30, 6)
const_by_tambon: (138, 7)
party_by_amphoe: (50, 4)
votes_per_station: (302, 5)


In [28]:
# Winner per amphoe (quick check)
winner_by_amphoe = (
    const_by_amphoe
    .sort_values('score', ascending=False)
    .groupby('amphoe')
    .first()
    .reset_index()
    [['amphoe', 'candidate_name', 'party_name', 'score']]
)
print('ผู้ชนะรายอำเภอ:')
print(winner_by_amphoe.to_string(index=False))

ผู้ชนะรายอำเภอ:
          amphoe       candidate_name     party_name  score
  ล่วงหน้านอกเขต นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง   4659
อำเภอทุ่งศรีอุดม นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง  10136
    อำเภอน้ำขุ่น นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง  10171
     อำเภอน้ำยืน นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง  25904
      อำเภอสำโรง นายสมศักดิ์ บุญประชม เพื่อไทรวมพลัง   9673


## 7 · Export Processed Data

In [29]:
# Raw tidy (station-level)
df_const.to_csv(OUT / 'constituency_station.csv', index=False, encoding='utf-8-sig')
df_party.to_csv(OUT / 'partylist_station.csv',    index=False, encoding='utf-8-sig')

# Aggregated
const_by_amphoe.to_csv(OUT / 'constituency_by_amphoe.csv', index=False, encoding='utf-8-sig')
const_by_tambon.to_csv(OUT / 'constituency_by_tambon.csv', index=False, encoding='utf-8-sig')
party_by_amphoe.to_csv(OUT / 'partylist_by_amphoe.csv',    index=False, encoding='utf-8-sig')
votes_per_station.to_csv(OUT / 'votes_per_station.csv',    index=False, encoding='utf-8-sig')

# Mapping lookups (for dashboard)
pd.DataFrame(cand_map_raw).to_csv(OUT / 'candidate_mapping.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(party_map_raw).to_csv(OUT / 'party_mapping.csv',    index=False, encoding='utf-8-sig')

print('Exported:')
for f in sorted(OUT.glob('*.csv')):
    print(f'  {f.name}')

Exported:
  candidate_mapping.csv
  constituency_by_amphoe.csv
  constituency_by_tambon.csv
  constituency_station.csv
  party_mapping.csv
  partylist_by_amphoe.csv
  partylist_station.csv
  votes_per_station.csv


## Summary

| File | Description |
|------|-------------|
| `constituency_station.csv` | Constituency scores — station level (tidy) |
| `partylist_station.csv` | Party-list scores — station level (tidy) |
| `constituency_by_amphoe.csv` | Constituency scores aggregated by amphoe |
| `constituency_by_tambon.csv` | Constituency scores aggregated by tambon |
| `partylist_by_amphoe.csv` | Party-list scores by amphoe (top 10 parties) |
| `votes_per_station.csv` | Total votes per station (for turnout analysis) |

**Next:** `02_turnout_analysis.ipynb`